# Midterm research: a preliminary context/history forecaster

This is **HeTCo-v0**, a project-defined feature-based prototype on a fixed Independent Leiden/GED reference, not full heterogeneous/temporal/consensus detection. All 2015 data were inspected. Negative results, zero-support classes and unresolved transitions are retained. The three original pair demos remain separate.

Run the research command documented in `docs/RUNBOOK.md` first. This notebook reads verified artifacts; it never silently retrains or discovers a different run.

In [ ]:
# Select the repository and run explicitly. Set these variables when launching
# Jupyter from another directory; selecting a Python kernel alone does not set cwd.
import os
from pathlib import Path
import sys
ROOT = Path(os.environ.get("REDDIT_SNA_PROJECT", Path.cwd())).resolve()
RUN = Path(os.environ.get("REDDIT_RESEARCH_RUN", ROOT / "artifacts/research/midterm-context-003")).resolve()
assert (ROOT / "pyproject.toml").is_file(), "Set REDDIT_SNA_PROJECT to the repository root"
assert RUN.is_relative_to(ROOT / "artifacts"), "Select a project artifact run"
print("Interpreter:", sys.executable)
print("Run:", RUN)
from reddit_sna.reporting.research import read_research, build_figures
from IPython.display import display, Markdown
get_ipython().run_line_magic("matplotlib", "inline")  # Render images even if the launcher uses Agg.
import matplotlib.pyplot as plt

plt.ioff()  # Explicit display avoids duplicate automatic figures.


In [ ]:
# The consumer checks file hashes, full sample/model/split keys, fit schedules,
# frozen protocol and metrics recomputed from predictions. It never unpickles.
delivery = read_research(RUN)
p = delivery["protocol"]
print("Status:", delivery["manifest"]["status"])
print("Protocol:", p["protocol_id"], "seeds:", p["seeds"])
print("Eligible samples:", len(delivery["samples"]))
print("Excluded transitions:", delivery["quality"]["unresolved"])
print("Exposure: inspected development data; human annotation pending; novelty unverified")


## Experimental plan and population

Each community at month t is predicted at t+1. S contains five structural features; H contains seven past-only history attributes; L contains accepted-predecessor structure/missing flags; C summarizes within-month corpus/thread context. No complete-future-thread participant counts are allowed. B5 and proposed use identical boosting parameters and sample/label/reference keys. A1 removes C, A4 removes membership dynamics, and A6 removes L.

Main label-month holdout starts in October 2015. Rolling diagnostics refit on earlier label months for July–December. Three model seeds are repeats, not independent test months.

In [ ]:
# Inspect the actual fixed settings before viewing any performance.
import json
print(json.dumps(p, indent=2))
figures = build_figures(delivery)
display(figures["population"])


## Baseline, proposed and ablation results

A preliminary method can fail to improve; completion does not require a positive delta. A1 and B5 must give identical predictions. A change of target labels is not evidence of better forecasting.

In [ ]:
# Values below come exclusively from independently verified prediction files.
rows = [r for r in delivery["evaluation"]["means"] if r["window"] == "main"]
text = "| Model | Six-class macro-F1 | Delta vs B5 | Samples/seed |\n|---|---:|---:|---:|\n"
text += "\n".join(f"| {r['model']} | {r['mean_macro_f1_six']:.6f} | {r['delta_vs_B5']:+.6f} | {r['n_per_seed']} |" for r in rows)
display(Markdown(text))
display(figures["comparison"])
display(figures["ablations"])


In [ ]:
# Track performance over time and retain class support/errors.
display(figures["monthly"])
display(figures["confusion"])
for row in delivery["evaluation"]["scores"]:
    if row["window"] == "main" and row["seed"] == 42 and row["model"] == "HeTCo_v0":
        print(json.dumps(row["per_class"], indent=2))


In [ ]:
# Context distributions can expose degenerate or sparse explanatory variables.
display(figures["context"])
print("Recorded fit count:", len(delivery["fits"]))
print("Native peak memory:", delivery["manifest"]["native_peak_memory_mb"])
print(delivery["manifest"]["memory_reason"])


## Uncertainty and interpretation limits

No interval is reported for the three-month primary holdout. The six-month moving-block exercise uses paired month blocks of length 2 and 1,000 resamples; seed scores are averaged inside each resample. It is exploratory, with limited time support, and cannot establish H3 or unseen generalization.

Human review, closest-work novelty, full T/temporal/consensus detection and a confirmed untouched period remain separate obligations. Synthetic harder-regime findings are in the group report and their own verified run.

In [ ]:
display(figures["uncertainty"])
print(json.dumps(delivery["evaluation"]["uncertainty"], indent=2))
# Close figures after display; notebook output retains the actual rendered images.
for figure in figures.values():
    plt.close(figure)
